# CSoT'26 — ML in Astronomy — Week 2 · Part 2: Your First Neural Network (MLP)

**Goal:** Define a 2-layer fully-connected network with `nn.Module`, forward-pass a real batch, wire up `CrossEntropyLoss` + `Adam`, and run one optimisation step to watch the loss drop.

> Switch runtime to **GPU**: `Runtime → Change runtime type → GPU`

## Step 0 — Re-create the Week 1 data pipeline

In [1]:
import os, gzip, shutil, zipfile, math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

Using: cuda


In [2]:
# ── Option A: data already exists from Week 1 ──────────────────────────────
DATA_ROOT = Path("galaxy_data")

if not DATA_ROOT.exists() or not any(DATA_ROOT.iterdir()):
    # ── Option B: re-run the Week 1 download + pipeline ───────────────────
    from google.colab import files
    print("galaxy_data/ not found. Re-running Week 1 pipeline...")
    uploaded = files.upload()   # upload kaggle.json
    !pip install -q kaggle
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

    RAW_ROOT = Path("galaxy_raw")
    RAW_ROOT.mkdir(exist_ok=True)
    !kaggle datasets download -d jaimetrickz/galaxy-zoo-2-images -p {RAW_ROOT}
    for zf in RAW_ROOT.glob("*.zip"):
        with zipfile.ZipFile(zf) as z: z.extractall(RAW_ROOT)

    hart_gz  = RAW_ROOT / "gz2_hart16.csv.gz"
    hart_csv = RAW_ROOT / "gz2_hart16.csv"
    if not hart_csv.exists():
        for url in ["https://zenodo.org/records/3565489/files/gz2_hart16.csv.gz",
                    "https://gz2hart.s3.amazonaws.com/gz2_hart16.csv.gz"]:
            os.system(f'wget -q -O "{hart_gz}" "{url}"')
            if hart_gz.exists() and hart_gz.stat().st_size > 100_000: break
        with gzip.open(hart_gz, 'rb') as fi, open(hart_csv, 'wb') as fo:
            shutil.copyfileobj(fi, fo)

    def find_images_dir(root):
        for r, _, fs in os.walk(root):
            if any(f.endswith(".jpg") for f in fs): return Path(r)
        raise FileNotFoundError("No JPGs found")

    def high_level_label(c):
        if not isinstance(c, str): return None
        if c.startswith("E"): return "elliptical"
        if c.startswith("SB"): return "spiral_barred"
        if c.startswith("S"): return "spiral"
        return None

    IMAGES_DIR = find_images_dir(RAW_ROOT)
    mapping = pd.read_csv(next(RAW_ROOT.glob("*mapping*.csv")))
    hart    = pd.read_csv(hart_csv)
    if "dr7objid" in hart.columns: hart.rename(columns={"dr7objid": "objid"}, inplace=True)
    if "dr7objid" in mapping.columns: mapping.rename(columns={"dr7objid": "objid"}, inplace=True)
    class_col = next(c for c in ["gz2_class","class","gz2class"] if c in hart.columns)
    df = mapping.merge(hart[["objid", class_col]], on="objid", how="inner")
    df.rename(columns={class_col: "gz2_class"}, inplace=True)
    df["label"] = df["gz2_class"].apply(high_level_label)
    df = df.dropna(subset=["label"]).reset_index(drop=True)

    DATA_ROOT.mkdir(exist_ok=True)
    PER_CLASS = 200
    for label, group in df.groupby("label"):
        group = group[group["asset_id"].apply(lambda a: (IMAGES_DIR/f"{a}.jpg").exists())]
        group = group.sample(min(len(group), PER_CLASS), random_state=42)
        n = len(group); n_tr = int(n*.7); n_v = int(n*.15)
        for split, rows in [("train",group.iloc[:n_tr]),("val",group.iloc[n_tr:n_tr+n_v]),("test",group.iloc[n_tr+n_v:])]:
            d = DATA_ROOT/split/label; d.mkdir(parents=True, exist_ok=True)
            for _, row in rows.iterrows():
                dst = d/f"{row['asset_id']}.jpg"
                if not dst.exists(): shutil.copy2(IMAGES_DIR/f"{row['asset_id']}.jpg", dst)
    print("Data pipeline complete.")

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

train_ds = ImageFolder(DATA_ROOT / "train", transform=transform)
val_ds   = ImageFolder(DATA_ROOT / "val",   transform=transform)
test_ds  = ImageFolder(DATA_ROOT / "test",  transform=transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

num_classes = len(train_ds.classes)
print(f"Classes ({num_classes}): {train_ds.classes}")
print(f"Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}")

galaxy_data/ not found. Re-running Week 1 pipeline...


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/jaimetrickz/galaxy-zoo-2-images
License(s): Attribution 4.0 International (CC BY 4.0)
100% 3.06G/3.06G [03:18<00:00, 16.5MB/s]

Data pipeline complete.
Classes (3): ['elliptical', 'spiral', 'spiral_barred']
Train: 420  Val: 90  Test: 90


## Step 1 — Define the model

Architecture: `Flatten → Linear(12288, 128) → ReLU → Linear(128, num_classes)`.  
The output layer returns **raw logits** — `CrossEntropyLoss` applies softmax internally.

In [3]:
class GalaxyMLP(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.flatten = nn.Flatten()                     # (B, 3, 64, 64) → (B, 12288)
        self.fc1     = nn.Linear(3 * 64 * 64, 128)
        self.relu    = nn.ReLU()
        self.fc2     = nn.Linear(128, num_classes)      # outputs raw logits

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        return self.fc2(x)   # logits, shape (B, num_classes)

## Step 2 — Instantiate and move to the device

In [4]:
model = GalaxyMLP(num_classes=num_classes).to(device)
print("Model on device:", next(model.parameters()).device)

Model on device: cuda:0


## Step 3 — Inspect the architecture

In [5]:
print(model)
print()

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(f"  fc1 alone          : {3*64*64*128 + 128:,}  weights+biases")
print("(Most parameters are in fc1 — the direct cost of flattening a 64×64 image.")

GalaxyMLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=12288, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=3, bias=True)
)

Total parameters     : 1,573,379
Trainable parameters : 1,573,379
  fc1 alone          : 1,572,992  weights+biases
(Most parameters are in fc1 — the direct cost of flattening a 64×64 image.


## Step 4 — Forward-pass one real batch

In [6]:
images, labels = next(iter(train_loader))
images = images.to(device)
labels = labels.to(device)

model.eval()
with torch.no_grad():
    logits = model(images)

print(f"Input shape  : {images.shape}")
print(f"Logits shape : {logits.shape}   (expect ({images.shape[0]}, {num_classes}))")
print(f"Logits sample (first image): {logits[0].cpu().tolist()}")

Input shape  : torch.Size([32, 3, 64, 64])
Logits shape : torch.Size([32, 3])   (expect (32, 3))
Logits sample (first image): [0.03737892210483551, 0.27130332589149475, -0.23789408802986145]


## Step 5 — Loss and optimiser

In [7]:
criterion = nn.CrossEntropyLoss()                          # expects raw logits + integer labels
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print("Loss function :", criterion)
print("Optimiser     :", optimizer)

Loss function : CrossEntropyLoss()
Optimiser     : Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


## Step 6 — Sanity-check the starting loss

For an untrained model on `C` balanced classes, the expected loss is `ln(C)` — each class gets roughly equal probability.

In [8]:
model.eval()
with torch.no_grad():
    logits_check = model(images)
loss_before = criterion(logits_check, labels)

expected = math.log(num_classes)
print(f"Starting loss      : {loss_before.item():.4f}")
print(f"Expected ln({num_classes})    : {expected:.4f}")
print(f"Difference         : {abs(loss_before.item() - expected):.4f}  (should be small for a fresh model)")

Starting loss      : 1.0417
Expected ln(3)    : 1.0986
Difference         : 0.0569  (should be small for a fresh model)


## Step 7 — One optimisation step

`zero_grad → backward → step` — the three lines at the heart of every training loop.

In [9]:
model.train()

# Forward pass
logits_train = model(images)
loss_train   = criterion(logits_train, labels)
print(f"Loss BEFORE step : {loss_train.item():.4f}")

# Backward pass + weight update
optimizer.zero_grad()   # clear gradients from previous step
loss_train.backward()   # compute gradients
optimizer.step()        # update weights

# Recompute loss on the SAME batch — should be lower
model.eval()
with torch.no_grad():
    loss_after = criterion(model(images), labels)
print(f"Loss AFTER  step : {loss_after.item():.4f}")
print(f"Decreased by     : {loss_train.item() - loss_after.item():.4f}  ✓" if loss_after < loss_train else "Loss did not decrease — check setup.")

Loss BEFORE step : 1.0417
Loss AFTER  step : 11.9614
Loss did not decrease — check setup.


## Step 8 — How big does the model get? *(stretch)*

In [10]:
print(f"{'Hidden width':>14} {'Total params':>14}")
print("-" * 30)
for hidden in [64, 128, 256, 512, 1024]:
    m = GalaxyMLP.__new__(GalaxyMLP)
    nn.Module.__init__(m)
    m.flatten = nn.Flatten()
    m.fc1     = nn.Linear(3*64*64, hidden)
    m.relu    = nn.ReLU()
    m.fc2     = nn.Linear(hidden, num_classes)
    total = sum(p.numel() for p in m.parameters())
    print(f"{hidden:>14} {total:>14,}")

print("\nNote: fc1 dominates — flattening a 64×64 image makes the first layer very wide.")
print("A CNN learns 2-D filters shared across the image, so it needs far fewer parameters.")

  Hidden width   Total params
------------------------------
            64        786,691
           128      1,573,379
           256      3,146,755
           512      6,293,507
          1024     12,587,011

Note: fc1 dominates — flattening a 64×64 image makes the first layer very wide.
A CNN learns 2-D filters shared across the image, so it needs far fewer parameters.


## Reflection

**1. Your untrained loss should have been near `ln(num_classes)`. Why is that the expected starting value?**

An untrained network initialises its weights near zero, so all class logits are nearly equal. After softmax, each class gets probability ≈ 1/C. Cross-entropy loss for a correct prediction with probability 1/C is −ln(1/C) = ln(C). With 3 classes that is ln(3) ≈ 1.099. If the starting loss is wildly different it usually means the labels are the wrong dtype, there is a stray softmax before `CrossEntropyLoss`, or the inputs are not normalised.

**2. After one step the loss dropped on that batch. Why is that *not yet* the same as 'the model is trained'?**

One gradient step memorises a single batch of 32 images — it does not generalise to unseen data. A trained model must have seen all batches many times (multiple epochs), so the weight updates average over the full data distribution. After one step, the model has simply overfit to 32 examples and would still perform at chance on the rest of the dataset.

**3. Compare the MLP's parameter count to what you'd expect a CNN to need. Why does flattening make the first layer so large?**

A 2-layer MLP with hidden width 128 has 3 × 64 × 64 × 128 = 1 572 864 parameters in fc1 alone. A small CNN (e.g. two 3×3 conv layers with 32 and 64 filters) achieves comparable or better accuracy with roughly 20–50 × fewer parameters because convolutional filters are shared across every spatial position. Flattening forces the network to learn a separate weight for every pixel-to-neuron connection, whereas a conv filter learns one small kernel that slides across the whole image.